# Tier 3 Solutions Set 2: Analysis and ML

Solutions for Assignment Set 2: Analysis and ML.

In [ ]:
import numpy as np
import math
from collections import defaultdict
from scipy import stats

---

## Problem 1: Promoter Motif Frequency (1 star)

In [ ]:
PROMOTER_MOTIFS = {
    'TATA_box': 'TATAAA',
    'GC_box':   'GGGCGG',
    'CAAT_box': 'CCAAT',
}


def analyze_promoter_motifs(sequences: list[str]) -> dict:
    """
    Analyze the frequency and position distribution of promoter motifs.

    Args:
        sequences: List of promoter sequences (each ~200 bp).

    Returns:
        Dict keyed by motif name, each value is a dict with:
        'count' (total occurrences), 'frequency' (fraction of seqs with >= 1 hit),
        'positions' (list of positions across all sequences).
    """
    results = {}
    for name, motif in PROMOTER_MOTIFS.items():
        total_count = 0
        seqs_with_motif = 0
        all_positions = []

        for seq in sequences:
            positions_in_seq = []
            start = 0
            while True:
                idx = seq.find(motif, start)
                if idx == -1:
                    break
                positions_in_seq.append(idx)
                start = idx + 1  # allow overlapping hits

            total_count += len(positions_in_seq)
            if positions_in_seq:
                seqs_with_motif += 1
            all_positions.extend(positions_in_seq)

        results[name] = {
            'count':     total_count,
            'frequency': seqs_with_motif / len(sequences),
            'positions': all_positions,
        }

    return results


# Test
np.random.seed(1)
bases = list('ACGT')

promoters = [''.join(np.random.choice(bases, 200)) for _ in range(10)]
for i in range(8):
    pos = np.random.randint(130, 160)
    promoters[i] = promoters[i][:pos] + 'TATAAA' + promoters[i][pos + 6:]
for i in range(5):
    pos = np.random.randint(50, 100)
    promoters[i] = promoters[i][:pos] + 'CCAAT' + promoters[i][pos + 5:]

results = analyze_promoter_motifs(promoters)
for name, data in results.items():
    print(f"{name}: count={data['count']}, frequency={data['frequency']:.2f}, "
          f"positions_sample={data['positions'][:5]}")

---

## Problem 2: Multiple Testing Correction (2 stars)

In [ ]:
def bonferroni_correction(pvalues: list[float]) -> list[float]:
    """
    Apply Bonferroni correction to a list of p-values.

    Args:
        pvalues: Raw p-values.

    Returns:
        Adjusted p-values in the same order as input.
    """
    n = len(pvalues)
    return [min(p * n, 1.0) for p in pvalues]


def bh_correction(pvalues: list[float]) -> list[float]:
    """
    Apply Benjamini-Hochberg FDR correction to a list of p-values.

    Args:
        pvalues: Raw p-values.

    Returns:
        Adjusted p-values (q-values) in the same order as input.
    """
    n = len(pvalues)
    # Sort ascending, tracking original positions
    order = sorted(range(n), key=lambda i: pvalues[i])
    adj = [0.0] * n

    for rank, idx in enumerate(order, start=1):
        adj[idx] = min(pvalues[idx] * n / rank, 1.0)

    # Enforce monotonicity: step backwards through sorted positions
    for i in range(len(order) - 2, -1, -1):
        adj[order[i]] = min(adj[order[i]], adj[order[i + 1]])

    return adj


# Test
np.random.seed(10)
null_pvals = list(np.random.uniform(0.1, 1.0, 90))
sig_pvals  = list(np.random.uniform(0.0001, 0.005, 10))
pvals = null_pvals + sig_pvals

bonf = bonferroni_correction(pvals)
bh   = bh_correction(pvals)

alpha = 0.05
print(f"Total tests: {len(pvals)}")
print(f"Significant (raw p < {alpha}):         {sum(p < alpha for p in pvals)}")
print(f"Significant (Bonferroni adj < {alpha}): {sum(p < alpha for p in bonf)}")
print(f"Significant (BH adj < {alpha}):         {sum(p < alpha for p in bh)}")

---

## Problem 3: k-NN Classifier (2 stars)

In [ ]:
def knn_predict(
    X_train: np.ndarray,
    y_train: list,
    X_test: np.ndarray,
    k: int = 3,
) -> list:
    """
    Classify test samples using k-nearest neighbors.

    Args:
        X_train: Training feature matrix, shape (n_train, n_features).
        y_train: Training labels, length n_train.
        X_test: Test feature matrix, shape (n_test, n_features).
        k: Number of nearest neighbors to consider.

    Returns:
        List of predicted labels for each test sample.
    """
    predictions = []
    for test_point in X_test:
        # Euclidean distances to all training points
        distances = np.sqrt(np.sum((X_train - test_point) ** 2, axis=1))
        # Indices of k nearest neighbors
        nn_indices = np.argsort(distances)[:k]
        nn_labels  = [y_train[i] for i in nn_indices]
        # Majority vote
        predictions.append(max(set(nn_labels), key=nn_labels.count))
    return predictions


def loocv_accuracy(
    X: np.ndarray,
    y: list,
    k: int = 3,
) -> float:
    """
    Estimate classification accuracy using leave-one-out cross-validation.

    Args:
        X: Feature matrix, shape (n_samples, n_features).
        y: Labels, length n_samples.
        k: Number of nearest neighbors.

    Returns:
        LOOCV accuracy (fraction of correct predictions).
    """
    n = len(y)
    correct = 0
    for i in range(n):
        X_train = np.delete(X, i, axis=0)
        y_train = y[:i] + y[i + 1:]
        pred = knn_predict(X_train, y_train, X[i:i + 1], k=k)
        if pred[0] == y[i]:
            correct += 1
    return correct / n


# Test
np.random.seed(99)
n_genes = 20

tumor  = np.random.randn(15, n_genes)
tumor[:, :10] += 2.0
normal = np.random.randn(15, n_genes)

X = np.vstack([tumor, normal])
y = ['tumor'] * 15 + ['normal'] * 15

acc  = loocv_accuracy(X, y, k=3)
acc5 = loocv_accuracy(X, y, k=5)
print(f"LOOCV accuracy (k=3): {acc:.3f}")
print(f"LOOCV accuracy (k=5): {acc5:.3f}")

---

## Problem 4: Feature Selection (2 stars)

In [ ]:
def variance_selection(X: np.ndarray, top_n: int = 50) -> list[int]:
    """
    Select top_n genes by variance across samples.

    Args:
        X: Expression matrix, shape (n_samples, n_genes).
        top_n: Number of genes to select.

    Returns:
        List of gene indices sorted by descending variance.
    """
    variances = np.var(X, axis=0)
    return list(np.argsort(variances)[::-1][:top_n])


def mutual_information_selection(
    X: np.ndarray,
    y: list,
    top_n: int = 50,
    n_bins: int = 4,
) -> list[int]:
    """
    Select top_n genes by mutual information with class label.

    Args:
        X: Expression matrix, shape (n_samples, n_genes).
        y: Class labels, length n_samples.
        top_n: Number of genes to select.
        n_bins: Number of quantile bins for discretizing expression values.

    Returns:
        List of gene indices sorted by descending mutual information.
    """
    unique_classes = list(set(y))
    n_samples, n_genes = X.shape
    mi_scores = []

    for g in range(n_genes):
        # Discretize gene expression into n_bins quantile bins
        quantiles = np.percentile(X[:, g], np.linspace(0, 100, n_bins + 1))
        bins = np.digitize(X[:, g], quantiles[1:-1])  # values 0..n_bins-1

        # Build joint and marginal distributions
        joint_counts: dict = defaultdict(int)
        x_counts:     dict = defaultdict(int)
        y_counts:     dict = defaultdict(int)

        for xi, yi in zip(bins, y):
            joint_counts[(xi, yi)] += 1
            x_counts[xi] += 1
            y_counts[yi] += 1

        mi = 0.0
        for (xi, yi), cnt in joint_counts.items():
            pxy = cnt / n_samples
            px  = x_counts[xi] / n_samples
            py  = y_counts[yi] / n_samples
            mi += pxy * math.log2(pxy / (px * py))

        mi_scores.append(mi)

    return list(np.argsort(mi_scores)[::-1][:top_n])


def jaccard_index(set_a: list[int], set_b: list[int]) -> float:
    """
    Compute Jaccard similarity between two sets.

    Returns:
        |A ∩ B| / |A ∪ B|
    """
    a, b = set(set_a), set(set_b)
    if not a and not b:
        return 1.0
    return len(a & b) / len(a | b)


# Test
np.random.seed(5)
n_samples, n_genes = 60, 200
y_labels = ['tumor'] * 30 + ['normal'] * 30

X_expr = np.random.randn(n_samples, n_genes)
X_expr[:30, :50] += 3.0

var_genes = variance_selection(X_expr, top_n=50)
mi_genes  = mutual_information_selection(X_expr, y_labels, top_n=50)

j = jaccard_index(var_genes, mi_genes)
overlap = set(var_genes) & set(mi_genes)
print(f"Variance top-50 sample: {sorted(var_genes)[:10]}")
print(f"MI top-50 sample:        {sorted(mi_genes)[:10]}")
print(f"Overlap size: {len(overlap)} genes")
print(f"Jaccard index: {j:.3f}")

---

## Problem 5: Random Forest for Cancer Classification (3 stars)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report


def train_cancer_classifier(
    X: np.ndarray,
    y: list,
    gene_names: list[str],
    n_estimators: int = 100,
    random_state: int = 42,
) -> dict:
    """
    Train and evaluate a Random Forest cancer classifier.

    Args:
        X: Expression matrix, shape (n_samples, n_genes).
        y: Cancer type labels.
        gene_names: List of gene names, length n_genes.
        n_estimators: Number of trees in the forest.
        random_state: Random seed.

    Returns:
        Dict with keys:
            'model': fitted RandomForestClassifier
            'accuracy': float
            'report': classification report string
            'top_genes': list of (gene_name, importance) for top 20 genes
    """
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=random_state, stratify=y
    )

    model = RandomForestClassifier(
        n_estimators=n_estimators, random_state=random_state, n_jobs=-1
    )
    model.fit(X_train, y_train)

    y_pred   = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    report   = classification_report(y_test, y_pred)

    importances = model.feature_importances_
    top_indices = np.argsort(importances)[::-1][:20]
    top_genes   = [(gene_names[i], float(importances[i])) for i in top_indices]

    return {
        'model':     model,
        'accuracy':  accuracy,
        'report':    report,
        'top_genes': top_genes,
    }


# Test
np.random.seed(42)
n_samples_per_class = 50
n_genes = 500

gene_names = [f'GENE_{i:04d}' for i in range(n_genes)]

lung   = np.random.randn(n_samples_per_class, n_genes)
breast = np.random.randn(n_samples_per_class, n_genes)
colon  = np.random.randn(n_samples_per_class, n_genes)

lung[:, :50]       += 3.0
breast[:, 100:150] += 3.0
colon[:, 200:250]  += 3.0

X_cancer = np.vstack([lung, breast, colon])
y_cancer = (['lung'] * n_samples_per_class + ['breast'] * n_samples_per_class +
            ['colon'] * n_samples_per_class)

results = train_cancer_classifier(X_cancer, y_cancer, gene_names)
print(f"Test accuracy: {results['accuracy']:.3f}\n")
print(results['report'])
print("Top 20 genes:")
for gene, imp in results['top_genes']:
    print(f"  {gene}: {imp:.4f}")

---

## Problem 6: Pharmacogenomics Variant Interpretation (3 stars)

In [ ]:
def generate_pgx_report(
    patient_variants: list[dict],
    drug_interactions: list[dict],
) -> list[dict]:
    """
    Generate a pharmacogenomics (PGx) clinical report for a patient.

    Matches patient variants to known drug-gene interactions.
    A drug interaction entry matches when:
      - The gene matches, AND
      - The interaction variant allele (e.g. '*4') appears in the patient's
        variant string (e.g. '*4/*4' or '*1/*4').

    Args:
        patient_variants: List of patient variant dicts with
            'gene', 'variant', 'zygosity'.
        drug_interactions: List of drug-gene interaction dicts with
            'gene', 'variant', 'drug', 'effect', 'recommendation', 'confidence'.

    Returns:
        Deduplicated list of report entries. Each entry contains:
        'drug', 'gene', 'effect', 'recommendation', 'confidence', 'zygosity'.
    """
    report = []
    seen: set = set()

    for pv in patient_variants:
        for interaction in drug_interactions:
            if interaction['gene'] != pv['gene']:
                continue
            # Check that the interaction allele appears in the patient's genotype
            if interaction['variant'] not in pv['variant']:
                continue

            key = (interaction['drug'], interaction['gene'])
            if key in seen:
                continue
            seen.add(key)

            report.append({
                'drug':           interaction['drug'],
                'gene':           interaction['gene'],
                'effect':         interaction['effect'],
                'recommendation': interaction['recommendation'],
                'confidence':     interaction['confidence'],
                'zygosity':       pv['zygosity'],
            })

    return report


# Test
patient_variants = [
    {'gene': 'CYP2D6',  'variant': '*4/*4',  'zygosity': 'homozygous'},
    {'gene': 'CYP2C19', 'variant': '*2/*17', 'zygosity': 'compound_heterozygous'},
    {'gene': 'TPMT',    'variant': '*1/*1',  'zygosity': 'homozygous_reference'},
    {'gene': 'DPYD',    'variant': '*2A/*1', 'zygosity': 'heterozygous'},
]

drug_interactions = [
    {'gene': 'CYP2D6',  'variant': '*4',  'drug': 'Codeine',
     'effect': 'poor_metabolizer',       'recommendation': 'avoid',                 'confidence': 'high'},
    {'gene': 'CYP2D6',  'variant': '*4',  'drug': 'Tamoxifen',
     'effect': 'reduced_efficacy',       'recommendation': 'consider_alternative',  'confidence': 'high'},
    {'gene': 'CYP2C19', 'variant': '*2',  'drug': 'Clopidogrel',
     'effect': 'reduced_activation',     'recommendation': 'avoid',                 'confidence': 'high'},
    {'gene': 'CYP2C19', 'variant': '*17', 'drug': 'Omeprazole',
     'effect': 'ultrarapid_metabolizer', 'recommendation': 'increase_dose',         'confidence': 'moderate'},
    {'gene': 'DPYD',    'variant': '*2A', 'drug': 'Fluorouracil',
     'effect': 'reduced_metabolism',     'recommendation': 'reduce_dose_50pct',     'confidence': 'high'},
    {'gene': 'TPMT',    'variant': '*3A', 'drug': 'Azathioprine',
     'effect': 'poor_metabolizer',       'recommendation': 'reduce_dose',           'confidence': 'high'},
]

report = generate_pgx_report(patient_variants, drug_interactions)
print(f"PGx report: {len(report)} actionable interactions found\n")
for entry in report:
    print(f"  Drug: {entry['drug']:<20} Gene: {entry['gene']:<10} "
          f"Effect: {entry['effect']:<25} Rec: {entry['recommendation']:<30} "
          f"Conf: {entry['confidence']}")

---

## Summary

Key implementation notes:

1. **Promoter Motifs**: Use `str.find` in a loop with `start = idx + 1` to catch overlapping hits
2. **BH Correction**: The monotonicity pass walks backwards through the rank-sorted order, not the original order
3. **k-NN**: `np.argsort` on Euclidean distances is the cleanest approach; `np.delete` for LOOCV fold creation
4. **Mutual Information**: Discretize via `np.digitize` on quantile boundaries; marginals computed from the same counts
5. **Random Forest**: Use `stratify=y` in `train_test_split` to maintain class balance across splits
6. **PGx**: Check variant allele membership with `in` operator on the genotype string; deduplicate by `(drug, gene)` key